In [ ]:
# Imports
import glob
import os
import re

import h5py
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"text.usetex": True})
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = "Times New Roman"
plt.rcParams["mathtext.fallback"] = "stix"
plt.style.use("seaborn-v0_8-deep")
COLORS  = plt.rcParams["axes.prop_cycle"].by_key()["color"]
MARKERS = ["o", "s", "^", "D"]

In [ ]:
# Parameters
batch    = "L32_OBC_m200_t11.0_g0.2_om5.0_spp10"  # folder name
base_dir = rf"C:\Users\liamc\Documents\majorana_chain_dynamics\results\{batch}"

i_ref     = 0    # reference site for correlation plots (0-based)

# Which file to load for all plots
g_plot    = 0.2
site_plot = 16

In [ ]:
# File discovery
def discover_files(base_dir):
    index = {}
    for fpath in glob.glob(os.path.join(base_dir, "**", "output.h5"), recursive=True):
        folder = os.path.basename(os.path.dirname(fpath))
        gm  = re.search(r"g(-?[\d.]+)", folder)
        pm  = re.search(r"p(even|odd)", folder)
        sm  = re.search(r"site(\d+)", folder)
        if gm and pm and sm:
            g       = float(gm.group(1))
            parity  = 1 if pm.group(1) == "even" else -1
            ac_site = int(sm.group(1))
            index[(g, parity, ac_site)] = fpath
    return index

index = discover_files(base_dir)
print(f"Found {len(index)} files")
for key in sorted(index):
    print(f"  g={key[0]}, parity={'even' if key[1]==1 else 'odd'}, ac_site={key[2]}")

In [ ]:
# Load one HDF5 file
def load_file(fpath):
    with h5py.File(fpath, "r") as f:
        d = {}
        # Metadata
        d["L"]                = int(f["L"][()])
        d["bc"]               = str(f["bc"][()])
        d["parity"]           = int(f["parity"][()])
        d["t1"]               = float(f["t1"][()])
        d["g"]                = float(f["g"][()])
        d["omega"]            = float(f["omega"][()])
        d["periods"]          = int(f["periods"][()])
        d["steps_per_period"] = int(f["steps_per_period"][()])
        d["ac_site"]          = int(f["ac_site"][()])
        d["E0_dmrg"]          = float(f["E0_dmrg"][()])
        # Time axes
        d["times"]      = f["times"][:]
        d["meas_times"] = f["meas_times"][:]
        # Scalar time series
        d["loschmidt"] = f["loschmidt"][:]
        d["energy_t"]  = f["energy_t"][:]
        d["energy_0"]  = f["energy_0"][:]
        d["maxdims"]   = f["maxdims"][:]
        # Autocorrelations
        d["autoc_x"] = f["re_autoc_x"][:] + 1j*f["im_autoc_x"][:]
        d["autoc_y"] = f["re_autoc_y"][:] + 1j*f["im_autoc_y"][:]
        d["autoc_z"] = f["re_autoc_z"][:] + 1j*f["im_autoc_z"][:]
        # Correlations: Julia writes (L, nmeas) and (L, L, nmeas)
        # so we transpose back to (nmeas, L) and (nmeas, L, L)
        d["X_t"]  = f["X_t"][:].T
        d["Y_t"]  = f["Y_t"][:].T
        d["Z_t"]  = f["Z_t"][:].T
        d["XX_t"] = f["XX_t"][:].transpose(2, 0, 1)
        d["YY_t"] = f["YY_t"][:].transpose(2, 0, 1)
        d["ZZ_t"] = f["ZZ_t"][:].transpose(2, 0, 1)
    # Connected correlations
    X, Y, Z = d["X_t"], d["Y_t"], d["Z_t"]
    d["XXc_t"] = d["XX_t"] - X[:, :, None] * X[:, None, :]
    d["YYc_t"] = d["YY_t"] - Y[:, :, None] * Y[:, None, :]
    d["ZZc_t"] = d["ZZ_t"] - Z[:, :, None] * Z[:, None, :]
    return d

# Load both parity sectors
d_even = load_file(index[(g_plot, 1,  site_plot)])
d_odd  = load_file(index[(g_plot, -1, site_plot)])
parities = [(d_even, "even", "-"), (d_odd, "odd", "--")]
T        = 2*np.pi / d_even["omega"]
t_over_T = d_even["times"] / T

In [ ]:
# Loschmidt echo and energy
for d_, p_label, ls in parities:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4), dpi=150)

    axes[0].plot(t_over_T, d_["loschmidt"], "-", color=COLORS[0], linewidth=1.5, marker='.', markersize=3)
    axes[0].set_xlabel(r"$t/T$", fontsize=12)
    axes[0].set_ylabel(r"$|\langle\psi(0)|\psi(t)\rangle|^2$", fontsize=12)
    axes[0].set_title("Loschmidt Echo", fontsize=13)
    axes[0].set_xlim(0, d_["periods"])
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(t_over_T, d_["energy_t"], "-",  color=COLORS[0], linewidth=1.5, marker='.', markersize=3, label=r"$\langle H(t)\rangle$")
    axes[1].plot(t_over_T, d_["energy_0"], "--", color=COLORS[1], linewidth=1.5, marker='.', markersize=3, label=r"$\langle H(0)\rangle$")
    axes[1].axhline(d_["E0_dmrg"], color="gray", linestyle=":", linewidth=1, label=r"$E_0$ (DMRG)")
    axes[1].set_xlabel(r"$t/T$", fontsize=12)
    axes[1].set_ylabel(r"$E$", fontsize=12)
    axes[1].set_title("Energy", fontsize=13)
    axes[1].set_xlim(0, d_["periods"])
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)

    plt.suptitle(
        rf"$g/t={d_['g']}$, $t_1={d_['t1']}$, $\omega={d_['omega']}$, $L={d_['L']}$ ({p_label} parity)",
        fontsize=13, y=1.02
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# Autocorrelation
ops    = ["X", "Y", "Z"]
site   = d_even["ac_site"]

for d_, p_label, ls in parities:
    autocs = [d_["autoc_x"], d_["autoc_y"], d_["autoc_z"]]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), dpi=150)
    for ax, op, autoc in zip(axes, ops, autocs):
        ax.plot(t_over_T, np.real(autoc), "-",  color=COLORS[0], linewidth=1.5, marker='.', markersize=3,
                label=rf"Re$\langle {op}_{{{site}}}(t){op}_{{{site}}}(0)\rangle$")
        ax.plot(t_over_T, np.imag(autoc), "--", color=COLORS[1], linewidth=1.5, marker='.', markersize=3,
                label=rf"Im$\langle {op}_{{{site}}}(t){op}_{{{site}}}(0)\rangle$")
        ax.set_xlabel(r"$t/T$", fontsize=12)
        ax.set_ylabel(rf"$\langle {op}_{{{site}}}(t){op}_{{{site}}}(0)\rangle$", fontsize=12)
        ax.set_title(rf"{op} Autocorrelation", fontsize=13)
        ax.set_xlim(0, d_["periods"])
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
    plt.suptitle(
        rf"site = {site}, $g/t={d_['g']}$, $t_1={d_['t1']}$, $\omega={d_['omega']}$, $L={d_['L']}$ ({p_label} parity)",
        fontsize=13, y=1.02
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation snapshot
snap_tT = 0     # change this to look at different times

snap_t    = snap_tT * T
meas_idx  = np.argmin(np.abs(d_even["meas_times"] - snap_t))
actual_tT = d_even["meas_times"][meas_idx] / T

L          = d_even["L"]
j_all      = np.arange(L)
mask       = j_all != i_ref
j_plot     = j_all[mask] + 1
site_label = i_ref + 1
ops        = ["X", "Y", "Z"]

for connected in [False, True]:
    kind  = "Connected Correlations" if connected else "Correlations"
    title = rf"$i={site_label}$, $t/T={actual_tT:.2f}$, $g/t={d_even['g']}$, $L={L}$"
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), dpi=150)
    for col, op in enumerate(ops):
        key   = f"{op}{op}c_t" if connected else f"{op}{op}_t"
        color = COLORS[0 if not connected else 1]
        if connected:
            ylabel = rf"$\langle {op}_{{{site_label}}} {op}_j\rangle - \langle {op}_{{{site_label}}}\rangle\langle {op}_j\rangle$"
        else:
            ylabel = rf"$\langle {op}_{{{site_label}}} {op}_j\rangle$"
        for d_, p_label, ls in parities:
            corr = d_[key][meas_idx]
            axes[col].plot(j_plot, corr[i_ref, mask],
                           linestyle=ls, color=color,
                           linewidth=1.5, label=f"{p_label}",
                           marker='.', markersize=3)
        axes[col].set_xlabel(r"Site $j$", fontsize=12)
        axes[col].set_ylabel(ylabel, fontsize=12)
        axes[col].set_title(rf"{op}-{op} {kind}", fontsize=12)
        axes[col].legend(fontsize=9)
        axes[col].grid(True, alpha=0.3)
        # axes[col].set_xscale("log")
        # axes[col].set_yscale("log")
    plt.suptitle(title, fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# Max bond dimension
fig, ax = plt.subplots(figsize=(8, 4), dpi=200)

for d_, p_label, ls in parities:
    ax.plot(t_over_T, d_["maxdims"], ls, color=COLORS[0], linewidth=1.5, label=f"{p_label} parity")

ax.set_xlabel(r"$t/T$", fontsize=12)
ax.set_ylabel(r"Max bond dimension", fontsize=12)
ax.set_title(rf"$g/t={d_even['g']}$, $\omega={d_even['omega']}$, $L={d_even['L']}$", fontsize=13)
ax.set_xlim(0, d_even["periods"])
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()